# 📖 Notebook 1: Full-Text Search Indexing

How does Facebook let you search through **trillions** of posts in under 500ms? The answer starts with a clever data structure called an **inverted index**.

In this notebook, we'll build search from scratch — starting with the slowest possible approach and working our way up to how production search engines actually work.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why naive `LIKE '%keyword%'` search is too slow at scale
- What an **inverted index** is and why it makes search fast
- How **tokenization** breaks text into searchable words
- How to use PostgreSQL full-text search (`tsvector`/`tsquery`)
- How to index and search documents with Elasticsearch
- The performance difference between each approach

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/fb-post-search
docker-compose up -d
```

Wait ~30 seconds for Elasticsearch to become healthy, then check:

```bash
curl http://localhost:9200/_cluster/health?pretty
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8081  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `fb_post_search`
- **Kibana** (Elasticsearch GUI): http://localhost:5601  
  Go to Dev Tools to run Elasticsearch queries interactively

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
from elasticsearch import Elasticsearch, helpers
import time
import json

# PostgreSQL connection
DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "fb_post_search",
    "user": "demo",
    "password": "demo"
}

# Elasticsearch connection
es = Elasticsearch("http://localhost:9200")

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM posts")
    count = cur.fetchone()[0]
    conn.close()
    print(f"✅ PostgreSQL connected — {count} posts in database")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    info = es.info()
    print(f"✅ Elasticsearch connected — version {info['version']['number']}")
except Exception as e:
    print(f"❌ Elasticsearch failed: {e}")
    print("   Run: docker-compose up -d (wait ~30s for ES to start)")

✅ PostgreSQL connected — 508 posts in database
✅ Elasticsearch connected — version 8.13.0


## 🐌 Approach 1: Naive SQL `LIKE` Search

The simplest way to search text in a database is with `LIKE`:

```sql
SELECT * FROM posts WHERE content LIKE '%coffee%';
```

This **works** — but it's painfully slow. Why?

The database has to read **every single row** and check if the content contains the word. This is called a **full table scan**. With 500 posts it's fine. With 3.6 trillion posts (Facebook scale), it's impossible.

Think of it like searching for a word in a book by reading every page. A book index (at the back) lets you jump straight to the right page. That's exactly what an inverted index does for search.

In [2]:
# Let's measure how slow LIKE search is

def search_with_like(keyword):
    """Search posts using SQL LIKE (full table scan)."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute(
        "SELECT id, content, like_count FROM posts WHERE content ILIKE %s ORDER BY like_count DESC LIMIT 10",
        (f"%{keyword}%",)
    )
    results = cur.fetchall()
    conn.close()
    return results

# Search for "coffee" and measure time
times = []
for _ in range(50):
    start = time.time()
    results = search_with_like("coffee")
    times.append((time.time() - start) * 1000)

print(f"🐌 SQL LIKE search for 'coffee' (50 runs):")
print(f"   Average: {sum(times)/len(times):.2f} ms")
print(f"   Results: {len(results)} posts found")
print()
for r in results[:3]:
    print(f"   [{r['like_count']} likes] {r['content'][:80]}...")

like_avg = sum(times) / len(times)

🐌 SQL LIKE search for 'coffee' (50 runs):
   Average: 17.16 ms
   Results: 10 posts found

   [4926 likes] New coffee shop opened downtown. The espresso is strong and the atmosphere is pe...
   [4572 likes] New coffee shop opened downtown. The espresso is strong and the atmosphere is pe...
   [4512 likes] New coffee shop opened downtown. The espresso is strong and the atmosphere is pe...


In [3]:
# Let's see the query plan — the database tells us exactly what it's doing

conn = get_db()
cur = conn.cursor()
cur.execute("EXPLAIN ANALYZE SELECT id, content FROM posts WHERE content ILIKE '%coffee%'")
plan = cur.fetchall()
conn.close()

print("📋 Query Execution Plan for LIKE search:")
print("=" * 60)
for row in plan:
    print(f"   {row[0]}")
print()
print("👆 Notice 'Seq Scan' — that means it reads EVERY row.")
print("   With millions of posts, this would take seconds or minutes.")

📋 Query Execution Plan for LIKE search:
   Seq Scan on posts  (cost=0.00..58.35 rows=26 width=109) (actual time=0.074..0.841 rows=27 loops=1)
     Filter: (content ~~* '%coffee%'::text)
     Rows Removed by Filter: 481
   Planning Time: 0.549 ms
   Execution Time: 0.873 ms

👆 Notice 'Seq Scan' — that means it reads EVERY row.
   With millions of posts, this would take seconds or minutes.


## 🧠 The Key Idea: Inverted Index

An **inverted index** flips the relationship between documents and words:

- **Normal**: For each post → list all the words it contains
- **Inverted**: For each word → list all the posts that contain it

```
Normal (how data is stored):                Inverted Index (what we build):
┌──────────┬─────────────────────┐          ┌──────────────┬────────────────┐
│ Post #1  │ "I love coffee"     │          │ "coffee"     │ [Post 1, 3, 5] │
│ Post #2  │ "Python is great"   │    →     │ "python"     │ [Post 2, 4]    │
│ Post #3  │ "Coffee and code"   │          │ "love"       │ [Post 1]       │
│ Post #4  │ "Learn Python"      │          │ "great"      │ [Post 2]       │
│ Post #5  │ "Best coffee shop"  │          │ "code"       │ [Post 3]       │
└──────────┴─────────────────────┘          │ "learn"      │ [Post 4]       │
                                            │ ...          │ ...            │
                                            └──────────────┴────────────────┘
```

Now searching for "coffee" is instant — just look up the key and get the list of post IDs!

### Tokenization

Before building the index, we need to break text into individual searchable words. This process is called **tokenization**. It typically includes:

1. **Lowercasing** — "Coffee" → "coffee"
2. **Removing punctuation** — "hello!" → "hello"
3. **Removing stop words** — drop common words like "the", "is", "a"
4. **Stemming** — "running" → "run", "quickly" → "quick"

In [4]:
# Let's build a simple inverted index in Python to understand the concept

import re
from collections import defaultdict

# Common English words that don't help with search
STOP_WORDS = {
    'the', 'a', 'an', 'is', 'it', 'in', 'on', 'at', 'to', 'for',
    'of', 'and', 'or', 'but', 'not', 'with', 'this', 'that', 'was',
    'are', 'be', 'has', 'had', 'have', 'do', 'does', 'did', 'from',
    'my', 'i', 'me', 'we', 'you', 'he', 'she', 'they', 'so', 'if'
}

def tokenize(text):
    """Break text into searchable tokens."""
    # 1. Lowercase
    text = text.lower()
    # 2. Extract words (letters and numbers only)
    words = re.findall(r'[a-z0-9]+', text)
    # 3. Remove stop words and very short words
    return [w for w in words if w not in STOP_WORDS and len(w) > 1]

# Example
sample = "Just finished building a REST API with Python and FastAPI!"
tokens = tokenize(sample)
print(f"Original: {sample}")
print(f"Tokens:   {tokens}")

Original: Just finished building a REST API with Python and FastAPI!
Tokens:   ['just', 'finished', 'building', 'rest', 'api', 'python', 'fastapi']


In [5]:
# Now build the full inverted index from all posts in our database

conn = get_db()
cur = conn.cursor()
cur.execute("SELECT id, content FROM posts")
all_posts = cur.fetchall()
conn.close()

# Build the inverted index: word → set of post IDs
inverted_index = defaultdict(set)

for post_id, content in all_posts:
    tokens = tokenize(content)
    for token in tokens:
        inverted_index[token].add(post_id)

print(f"📚 Built inverted index:")
print(f"   Posts indexed: {len(all_posts)}")
print(f"   Unique words:  {len(inverted_index)}")
print()

# Show some examples
for word in ['coffee', 'python', 'taylor', 'search']:
    post_ids = inverted_index.get(word, set())
    print(f"   '{word}' → {len(post_ids)} posts (IDs: {sorted(list(post_ids))[:5]}...)")

📚 Built inverted index:
   Posts indexed: 508
   Unique words:  1063

   'coffee' → 27 posts (IDs: [54, 55, 56, 57, 58]...)
   'python' → 23 posts (IDs: [1, 78, 98, 118, 138]...)
   'taylor' → 4 posts (IDs: [26, 51, 52, 53]...)
   'search' → 27 posts (IDs: [7, 41, 42, 46, 64]...)


In [6]:
# Search using our inverted index — should be nearly instant!

def search_inverted_index(keyword):
    """Look up posts using the inverted index."""
    token = keyword.lower()
    return inverted_index.get(token, set())

times_inv = []
for _ in range(50):
    start = time.time()
    post_ids = search_inverted_index("coffee")
    times_inv.append((time.time() - start) * 1000)

inv_avg = sum(times_inv) / len(times_inv)
print(f"⚡ Inverted index search for 'coffee' (50 runs):")
print(f"   Average: {inv_avg:.4f} ms")
print(f"   Posts found: {len(post_ids)}")
print()
print(f"🚀 Speedup vs LIKE: {like_avg / inv_avg:.0f}× faster!")
print()
print("💡 The inverted index is just a dictionary lookup — O(1) instead of O(n).")
print("   This is the fundamental idea behind every search engine.")

⚡ Inverted index search for 'coffee' (50 runs):
   Average: 0.0003 ms
   Posts found: 27

🚀 Speedup vs LIKE: 50691× faster!

💡 The inverted index is just a dictionary lookup — O(1) instead of O(n).
   This is the fundamental idea behind every search engine.


## 🐘 Approach 2: PostgreSQL Full-Text Search

PostgreSQL has **built-in** full-text search using two special types:

- `tsvector` — a processed version of text (tokenized, stemmed, weighted)
- `tsquery` — a search query that matches against tsvectors

Think of `tsvector` as PostgreSQL's built-in inverted index. It does the tokenization, stop word removal, and stemming automatically.

Let's add a full-text search column to our posts table and see how it compares.

In [7]:
# Add a tsvector column and GIN index to our posts table

conn = get_db()
cur = conn.cursor()

# Add the tsvector column (stores pre-processed search tokens)
cur.execute("ALTER TABLE posts ADD COLUMN IF NOT EXISTS search_vector tsvector")

# Populate it from the content column
cur.execute("UPDATE posts SET search_vector = to_tsvector('english', content)")

# Create a GIN index — this is what makes searches fast
# GIN = Generalized Inverted Index (it's literally an inverted index!)
cur.execute("CREATE INDEX IF NOT EXISTS idx_posts_search ON posts USING GIN(search_vector)")

conn.commit()
conn.close()

print("✅ Added tsvector column and GIN index to posts table")
print()
print("What just happened:")
print("  1. Added a 'search_vector' column that stores tokenized text")
print("  2. Populated it with to_tsvector() — handles tokenization + stemming")
print("  3. Created a GIN index — PostgreSQL's inverted index implementation")

✅ Added tsvector column and GIN index to posts table

What just happened:
  1. Added a 'search_vector' column that stores tokenized text
  2. Populated it with to_tsvector() — handles tokenization + stemming
  3. Created a GIN index — PostgreSQL's inverted index implementation


In [8]:
# Let's see what tsvector looks like

conn = get_db()
cur = conn.cursor()

cur.execute("SELECT content, search_vector FROM posts WHERE id = 1")
content, vector = cur.fetchone()
conn.close()

print("Original text:")
print(f"  {content}")
print()
print("tsvector (tokenized + stemmed):")
print(f"  {vector}")
print()
print("Notice how:")
print("  - 'building' became 'build' (stemming)")
print("  - 'the', 'a', 'with' are removed (stop words)")
print("  - Each token has position numbers for phrase matching")

Original text:
  Just finished building a REST API with Python and FastAPI. The automatic docs are amazing! 🚀

tsvector (tokenized + stemmed):
  'amaz':15 'api':6 'automat':12 'build':3 'doc':13 'fastapi':10 'finish':2 'python':8 'rest':5

Notice how:
  - 'building' became 'build' (stemming)
  - 'the', 'a', 'with' are removed (stop words)
  - Each token has position numbers for phrase matching


In [9]:
# Search with PostgreSQL full-text search

def search_with_tsquery(keyword):
    """Search posts using PostgreSQL full-text search."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute(
        """SELECT id, content, like_count,
                  ts_rank(search_vector, query) AS rank
           FROM posts, plainto_tsquery('english', %s) query
           WHERE search_vector @@ query
           ORDER BY rank DESC
           LIMIT 10""",
        (keyword,)
    )
    results = cur.fetchall()
    conn.close()
    return results

# Measure performance
times_ts = []
for _ in range(50):
    start = time.time()
    results = search_with_tsquery("coffee")
    times_ts.append((time.time() - start) * 1000)

ts_avg = sum(times_ts) / len(times_ts)

print(f"🐘 PostgreSQL tsquery search for 'coffee' (50 runs):")
print(f"   Average: {ts_avg:.2f} ms")
print(f"   Results: {len(results)}")
print()
for r in results[:3]:
    print(f"   [rank={r['rank']:.4f}, {r['like_count']} likes] {r['content'][:70]}...")

🐘 PostgreSQL tsquery search for 'coffee' (50 runs):
   Average: 20.62 ms
   Results: 10

   [rank=0.0760, 1234 likes] Tried quitting coffee for a month. Worst month of my life. Coffee and ...
   [rank=0.0760, 678 likes] Coffee is fuel for programmers. My morning ritual: brew coffee, open t...
   [rank=0.0760, 345 likes] The best coffee shop in San Francisco serves the most amazing pour-ove...


In [10]:
# Verify it uses the GIN index (not a full table scan)
#
# Note: with only ~500 rows the planner often prefers a sequential scan
# (reading the whole table is cheaper than loading the index). We disable
# seqscan for this session to force the planner to use our GIN index so
# you can see what the production query plan looks like.

conn = get_db()
cur = conn.cursor()
cur.execute("SET LOCAL enable_seqscan = OFF")
cur.execute(
    """EXPLAIN ANALYZE
       SELECT id FROM posts
       WHERE search_vector @@ plainto_tsquery('english', 'coffee')"""
)
plan = cur.fetchall()
conn.close()

print("📋 Query plan for tsquery search (with seqscan disabled):")
print("=" * 60)
for row in plan:
    print(f"   {row[0]}")
print()
print("👆 Notice 'Bitmap Index Scan on idx_posts_search'")
print("   The GIN index lets PostgreSQL jump straight to matching rows!")
print("   (On our tiny 500-row table the planner would normally skip the")
print("    index; at millions of rows it always wins.)")

📋 Query plan for tsquery search (with seqscan disabled):
   Bitmap Heap Scan on posts  (cost=76.69..122.10 rows=27 width=4) (actual time=0.199..0.211 rows=27 loops=1)
     Recheck Cond: (search_vector @@ '''coffe'''::tsquery)
     Heap Blocks: exact=39
     ->  Bitmap Index Scan on idx_posts_search  (cost=0.00..76.69 rows=27 width=0) (actual time=0.172..0.172 rows=54 loops=1)
           Index Cond: (search_vector @@ '''coffe'''::tsquery)
   Planning Time: 0.876 ms
   Execution Time: 0.233 ms

👆 Notice 'Bitmap Index Scan on idx_posts_search'
   The GIN index lets PostgreSQL jump straight to matching rows!
   (On our tiny 500-row table the planner would normally skip the
    index; at millions of rows it always wins.)


## 🔍 Approach 3: Elasticsearch

Elasticsearch is a **dedicated search engine** built on Apache Lucene. It's what companies like Facebook, Uber, and Netflix use when PostgreSQL full-text search isn't enough.

Why use Elasticsearch over PostgreSQL full-text search?

| Feature | PostgreSQL | Elasticsearch |
|---------|-----------|---------------|
| Relevance scoring | Basic (ts_rank) | Advanced (BM25) |
| Fuzzy matching | Limited | Built-in |
| Autocomplete | Manual | Built-in suggesters |
| Horizontal scaling | Hard | Built-in sharding |
| Analyzers | English only | 30+ languages |
| Real-time indexing | Triggers needed | Native |

**Rule of thumb**: Use PostgreSQL full-text search for simple cases (blog search, small datasets). Use Elasticsearch when search is a core feature of your product.

### How Elasticsearch Works

1. You create an **index** (like a database table)
2. You define **mappings** (like a schema — what fields exist and how to analyze them)
3. You **index documents** (insert data)
4. You **search** using a powerful query DSL

In [11]:
# Step 1: Create an Elasticsearch index with custom mappings

INDEX_NAME = "posts"

# Delete the index if it already exists (for re-running this notebook)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

# Define the index mapping
es.indices.create(
    index=INDEX_NAME,
    body={
        "settings": {
            "number_of_shards": 1,      # single shard for this demo
            "number_of_replicas": 0,     # no replicas for local dev
            "analysis": {
                "analyzer": {
                    "post_analyzer": {
                        "type": "custom",
                        "tokenizer": "standard",   # splits on whitespace/punctuation
                        "filter": ["lowercase", "stop", "snowball"]  # lowercase → remove stop words → stem
                    }
                }
            }
        },
        "mappings": {
            "properties": {
                "content": {
                    "type": "text",
                    "analyzer": "post_analyzer"     # use our custom analyzer
                },
                "user_id":    {"type": "integer"},
                "like_count": {"type": "integer"},
                "created_at": {"type": "date"}
            }
        }
    }
)

print("✅ Created Elasticsearch index 'posts'")
print()
print("The analyzer pipeline:")
print('  "I love Coffee!" → ["i", "love", "coffee"] → ["love", "coffee"] → ["love", "coffe"]')
print("   (tokenize)         (lowercase)              (stop words)         (stemming)")

✅ Created Elasticsearch index 'posts'

The analyzer pipeline:
  "I love Coffee!" → ["i", "love", "coffee"] → ["love", "coffee"] → ["love", "coffe"]
   (tokenize)         (lowercase)              (stop words)         (stemming)


In [12]:
# Step 2: Bulk-index all posts from PostgreSQL into Elasticsearch

conn = get_db()
cur = conn.cursor()
cur.execute("SELECT id, user_id, content, like_count, created_at FROM posts")
rows = cur.fetchall()
conn.close()

# Build bulk actions
actions = []
for row in rows:
    actions.append({
        "_index": INDEX_NAME,
        "_id": row[0],
        "_source": {
            "user_id": row[1],
            "content": row[2],
            "like_count": row[3],
            "created_at": row[4].isoformat() if row[4] else None
        }
    })

# Bulk index (much faster than indexing one at a time)
start = time.time()
success, errors = helpers.bulk(es, actions)
elapsed = time.time() - start

# Refresh so documents are immediately searchable
es.indices.refresh(index=INDEX_NAME)

print(f"✅ Indexed {success} posts into Elasticsearch in {elapsed:.2f}s")
if errors:
    print(f"❌ {len(errors)} errors occurred")

✅ Indexed 508 posts into Elasticsearch in 0.03s


In [13]:
# Step 3: Search with Elasticsearch

def search_elasticsearch(keyword):
    """Search posts using Elasticsearch."""
    result = es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "match": {
                    "content": keyword
                }
            },
            "size": 10
        }
    )
    return result["hits"]["hits"]

# Measure performance
times_es = []
for _ in range(50):
    start = time.time()
    results = search_elasticsearch("coffee")
    times_es.append((time.time() - start) * 1000)

es_avg = sum(times_es) / len(times_es)

print(f"🔍 Elasticsearch search for 'coffee' (50 runs):")
print(f"   Average: {es_avg:.2f} ms")
print(f"   Results: {len(results)} (showing top 3):")
print()
for hit in results[:3]:
    score = hit['_score']
    src = hit['_source']
    print(f"   [score={score:.4f}, {src['like_count']} likes] {src['content'][:70]}...")

🔍 Elasticsearch search for 'coffee' (50 runs):
   Average: 1.06 ms
   Results: 10 (showing top 3):

   [score=4.1115, 678 likes] Coffee is fuel for programmers. My morning ritual: brew coffee, open t...
   [score=4.0231, 345 likes] The best coffee shop in San Francisco serves the most amazing pour-ove...
   [score=4.0231, 1234 likes] Tried quitting coffee for a month. Worst month of my life. Coffee and ...


## 📊 Performance Comparison

Let's put all three approaches side by side.

In [14]:
import pandas as pd

comparison = pd.DataFrame([
    {"Approach": "SQL LIKE (full scan)", "Avg Latency (ms)": round(like_avg, 2),
     "How It Works": "Reads every row", "Scales?": "❌ No"},
    {"Approach": "PostgreSQL tsquery", "Avg Latency (ms)": round(ts_avg, 2),
     "How It Works": "GIN inverted index", "Scales?": "⚠️ Single node"},
    {"Approach": "Elasticsearch", "Avg Latency (ms)": round(es_avg, 2),
     "How It Works": "Distributed inverted index", "Scales?": "✅ Shards + replicas"},
])

print("📊 Search Performance Comparison")
print("=" * 80)
print(comparison.to_string(index=False))
print()
print("💡 Key takeaway: All indexed approaches use inverted indexes under the hood.")
print("   Elasticsearch wins at scale because it can distribute the index across nodes.")

📊 Search Performance Comparison
            Approach  Avg Latency (ms)               How It Works             Scales?
SQL LIKE (full scan)             17.16            Reads every row                ❌ No
  PostgreSQL tsquery             20.62         GIN inverted index      ⚠️ Single node
       Elasticsearch              1.06 Distributed inverted index ✅ Shards + replicas

💡 Key takeaway: All indexed approaches use inverted indexes under the hood.
   Elasticsearch wins at scale because it can distribute the index across nodes.


## 🔬 Understanding Elasticsearch Scoring

Elasticsearch doesn't just find matching documents — it **ranks** them by relevance using the **BM25** algorithm. Let's peek inside to understand what's happening.

In [15]:
# Use the 'explain' parameter to see how Elasticsearch scores each result

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {"match": {"content": "coffee"}},
        "explain": True,
        "size": 2
    }
)

for hit in result["hits"]["hits"]:
    print(f"Post {hit['_id']}: score = {hit['_score']:.4f}")
    print(f"  Content: {hit['_source']['content'][:80]}...")
    # Show the top-level explanation
    explanation = hit['_explanation']
    print(f"  Scoring: {explanation['description']}")
    print(f"  Value:   {explanation['value']:.4f}")
    print()

print("💡 BM25 considers:")
print("   - Term frequency (TF): How often 'coffee' appears in this post")
print("   - Inverse document frequency (IDF): How rare 'coffee' is across all posts")
print("   - Field length: Shorter documents with the term score higher")
print()
print("   We'll explore ranking in depth in Notebook 3!")

Post 55: score = 4.1115
  Content: Coffee is fuel for programmers. My morning ritual: brew coffee, open terminal, s...
  Scoring: weight(content:coffe in 47) [PerFieldSimilarity], result of:
  Value:   4.1115

Post 54: score = 4.0231
  Content: The best coffee shop in San Francisco serves the most amazing pour-over coffee. ...
  Scoring: weight(content:coffe in 46) [PerFieldSimilarity], result of:
  Value:   4.0231

💡 BM25 considers:
   - Term frequency (TF): How often 'coffee' appears in this post
   - Inverse document frequency (IDF): How rare 'coffee' is across all posts
   - Field length: Shorter documents with the term score higher

   We'll explore ranking in depth in Notebook 3!


## 🧹 Cleanup

In [16]:
# The Elasticsearch index will persist for Notebooks 2 and 3.
# Only run this if you want a fresh start:

# es.indices.delete(index=INDEX_NAME)
# print("🧹 Deleted Elasticsearch index")

print("✅ Index 'posts' kept for use in Notebooks 2 and 3.")
print("   To start fresh: uncomment the delete line above and re-run.")

✅ Index 'posts' kept for use in Notebooks 2 and 3.
   To start fresh: uncomment the delete line above and re-run.


## 📚 Summary

### Key Takeaways

1. **`LIKE` search is a full table scan** — it reads every row. Fine for 500 rows, impossible for billions.
2. **Inverted indexes flip the problem** — instead of scanning every document, you look up which documents contain the word. This is O(1) instead of O(n).
3. **Tokenization matters** — lowercasing, stop words, and stemming ensure users find what they're looking for regardless of how they type it.
4. **PostgreSQL has built-in full-text search** — `tsvector` + GIN index. Good enough for many applications.
5. **Elasticsearch is built for search at scale** — distributed inverted indexes, advanced scoring (BM25), and built-in features for autocomplete, fuzzy matching, etc.

### How This Maps to the Facebook Post Search Design

In the system design interview:
- Posts are **tokenized** when created (in the Ingestion Service)
- Tokens are stored in an **inverted index** (Redis sorted sets or Elasticsearch)
- Search queries look up the index to find matching post IDs
- Results are ranked and returned to the user

### Next Up

In **Notebook 2**, we'll build **typeahead and autocomplete** — showing suggestions as the user types, before they even finish their query.